[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/quantum/kitaev_wire/kitaev_wire.ipynb)

# Moving a Majorana boundary mode

A superconducting wire can support a mode at each end of a topological segment. Their separation suppresses the energy splitting of a shared fermionic degree of freedom. Gates move those ends. A sufficiently slow sweep carries a mode along; a fast sweep loses it to the bulk. We build the wire's couplings, solve its boundary modes, and evolve one of them while the gate moves.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np
from numpy.typing import ArrayLike

from numga import Algebra, NumpyContext, stack
from numga.sparse import SparseExtensor
from examples.animation import save_figure, show_animation
from examples.quantum.kitaev_wire import render
from examples.quantum.kitaev_wire.core import smooth_progress

In [ ]:
ga = Algebra("x+y+")
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()
# x and y label the two Majorana components at each site, not spatial directions.
Majorana = ga.gatype.vector()

Majorana generators square to one and anticommute: they form a Clifford basis. A quadratic fermionic evolution is generated by a bivector, whose commutator acts as a skew map on these directions. Here each sparse cell maps two real components at an input site to two at an output site. The field map assembles their action over the wire.

In Kitaev's Majorana notation, with $c_{2j-1}$ and $c_{2j}$ the two components at site $j$, the Hamiltonian reads $H = \tfrac{i}{2}\sum_j\left[-\mu\,c_{2j-1}c_{2j} + (t+\Delta)\,c_{2j}c_{2j+1} + (\Delta-t)\,c_{2j-1}c_{2j+2}\right]$.

In [ ]:
site_count = 96
sites = np.arange(site_count)
site_index = mv.scalar(sites[:, None]).field()           # Scalar[sites]
hopping = 1.0
pairing = 0.8
outside = 3.5
inside = 1.4
width = 1.2
left = 18.0
right = 82.0


def gate(sites: np.ndarray, left: ArrayLike, right: ArrayLike,
         inside: ArrayLike, outside: float, width: float) -> Scalar:
    # A smooth well in chemical potential, with the exterior held fixed.
    window = (np.tanh((sites - np.asarray(left)[..., None]) / width)
              - np.tanh((sites - np.asarray(right)[..., None]) / width)) / 2
    return mv.scalar((outside + (np.asarray(inside)[..., None] - outside) * window)[..., None]).field()

# Hopping and p-wave pairing add in one cross-site coupling and subtract in the other.
# A directed bond reads a component on the next site and returns one on this site.
bond = ((hopping + pairing) * mv.y * (mv.x | Majorana)
        + (pairing - hopping) * mv.x * (mv.y | Majorana))
forward = SparseExtensor(bond.broadcast_to((site_count - 1,)),
                         sites[:-1], sites[1:], (site_count, site_count))
bonds = forward - forward.adjoint()                       # skew field map


def generator(potential: Scalar) -> SparseExtensor:
    # Chemical potential turns the two components on the same site.
    local = -potential * mv.xy.commutator(Majorana)
    return bonds + SparseExtensor.from_diagonal(local)


def modes(potential: Scalar, count: int) -> tuple[Scalar, Majorana]:
    turning = generator(potential)
    # Squaring the norm of the turning map gives squared excitation energies,
    # against the identity on the wire's sites as metric.
    _, states = turning.adjoint()(turning).eigh(0 * turning + Majorana, count)
    energies = turning[..., None](states).scalar_norm_squared().sites.sum().square_root()
    return energies, states

Lower the chemical potential inside the gate from the outside value and solve for the lowest excitations at each step: their energies on the left, the weight of the lowest pair along the wire on the right.

In [ ]:
spectral_samples = 100
energy_count = 8

chemical_potentials = np.linspace(outside, inside, spectral_samples)
potential = gate(sites, left, right, chemical_potentials, outside, width)
energies, states = modes(potential, energy_count)          # [samples, modes] Scalar, [samples, modes] Majorana[sites]
# Each excitation has two real quadratures. Their combined weight is basis-independent.
density = states[:, :2].scalar_norm_squared().mean(axis=-1)
save_figure(render.draw_formation(chemical_potentials, energies, density, hopping), "kitaev_formation");

Lowering the chemical potential inside the gate creates a topological segment. For a uniform Kitaev chain with nonzero pairing, the transition is at $|\mu|=2t$; the dotted line marks that bulk threshold. Two low-energy modes separate from the other excitations and concentrate at the segment's boundaries. To resolve left and right, take the mean site index between every pair of the two modes, a symmetric map on their two coefficients, and combine the modes along its eigenvectors.

In [ ]:
destination = 48.0
steps = 720
stride = 6
progress = np.arange(steps // stride + 1) * stride / steps


def gate_at(progress: np.ndarray) -> Scalar:
    # The left boundary moves toward its destination along a smooth ramp.
    left_boundary = left + (destination - left) * smooth_progress(progress)
    return gate(sites, left_boundary, right, inside, outside, width)


def localize(states: Majorana) -> Majorana:
    # The site-index observable between every pair of the two modes.
    gram = states[..., :, None].scalar_product(site_index * states[..., None, :]).sites.sum()  # [..., modes, modes] Scalar
    position = (gram * Scalar).field(0, 1)                # Scalar[modes] <- Scalar[modes]
    # The eigenvectors of this small map select the left and right combinations.
    _, directions = position.eigh()
    return (directions.batch() * states[..., None, :]).sum(axis=-1)


potentials = gate_at(progress)
_, boundary_modes = modes(potentials, 2)                  # [frames, modes] Majorana[sites]
localized = localize(boundary_modes)                     # [frames, ends] Majorana[sites]

Evolve the left boundary mode while the gate moves, once slowly and once fast. Each step is a Cayley step of the turning map, with the gate taken halfway through the step.

In [ ]:
durations = np.array([180.0, 12.0])
duration_ms = 80
midpoint_potentials = gate_at((np.arange(steps) + 0.5) / steps)


def evolution(initial: Majorana) -> Iterator[Majorana]:
    state = initial.broadcast_to((len(durations),))
    dt = durations / steps
    yield state
    for block in midpoint_potentials.reshape(-1, stride):
        for potential in block:
            half_step = generator(potential) * (dt[:, None] / 2)
            # A Cayley step of a skew map is orthogonal: it preserves the mode norm.
            # Evaluating the gate halfway through the step gives second-order evolution.
            state = (Majorana - half_step).solve(state + half_step(state))
        yield state


history = stack(evolution(localized[0, 0]))              # [frames, speeds] Majorana[sites]
# Project the evolving field onto both instantaneous boundary modes.
overlap = history[:, :, None].scalar_product(boundary_modes[:, None]).sites.sum()
share = overlap.squared().sum(axis=-1)                    # [frames, speeds] Scalar
show_animation(render.animate_transport(progress, potentials, history, localized[:, 0], share, hopping),
               "kitaev_transport", duration_ms)

Both sweeps follow the same gate path, in different physical times: the slow sweep in the upper panel, the fast sweep in the middle one. The dashed profile is the instantaneous localized mode; the solid profile is actually evolved. The pale region marks where the local chemical potential lies in the topological regime. The slow sweep follows the boundary, while the fast sweep releases a spreading field. Bulk weight is the share outside the instantaneous two-dimensional boundary-mode space. The figure below shows both evolutions whole, the slow sweep on the left, with the gate's topological threshold dashed.

In [ ]:
save_figure(render.draw_transport(progress, history, potentials, hopping), "kitaev_transport");

Bring the boundaries together and their modes overlap. The left and right combinations then couple, lifting the lowest excitation away from zero. The curve below is a sequence of stationary eigenproblems, not another time evolution; beside it are the two localized modes at three separations. Once the overlap is appreciable, the localized combinations are not individually energy eigenstates.

In [ ]:
separations = np.linspace(4.0, 32.0, 100)
profile_separations = np.array([24.0, 12.0, 5.0])

closing_potentials = gate(sites, right - separations, right, inside, outside, width)
closing_energies, _ = modes(closing_potentials, 2)
splitting = closing_energies.mean(axis=-1)
profile_potentials = gate(sites, right - profile_separations, right, inside, outside, width)
_, profile_modes = modes(profile_potentials, 2)
save_figure(render.draw_overlap(separations, profile_separations, splitting, profile_potentials,
                                localize(profile_modes), hopping), "kitaev_overlap");

The plotted weight is squared Majorana-mode amplitude, not charge density or a quasiparticle population. This is a clean spinless Kitaev chain with prescribed real p-wave pairing and externally controlled chemical potential; it does not solve for the superconducting order parameter or include interactions and dissipation. [Kitaev's paper](https://arxiv.org/abs/cond-mat/0010440) gives the chain and its boundary modes. [Alicea and collaborators](https://arxiv.org/abs/1006.4395) develop gate-controlled transport and exchange through wire networks.